# Sierra_Leone - EDA

In [1]:
from pathlib import Path
import os

p = Path.cwd()
while not (p / "data").exists():
    p = p.parent
os.chdir(p)

print("Working directory set to:", os.getcwd())


Working directory set to: C:\KAIM Week-0\solar-challenge-week1


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scripts.eda_utils import zscore_flags, impute_median

# Settings
pd.set_option("display.max_columns", 100)
plt.rcParams.update({"figure.figsize": (10, 5)})

# File paths (update if needed)
RAW_PATH = "data/sierra_leone.csv"   # put your raw file here
CLEAN_PATH = "data/sierra_leone_clean.csv"

In [3]:
# Load data
df = pd.read_csv(RAW_PATH, parse_dates=["Timestamp"])
df.sort_values("Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
0,2021-10-30 00:01:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.1,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
1,2021-10-30 00:02:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
2,2021-10-30 00:03:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.2,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN
3,2021-10-30 00:04:00,-0.7,0.0,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.1,22.3,22.6,NaN
4,2021-10-30 00:05:00,-0.7,-0.1,-0.8,0.0,0.0,21.9,99.3,0.0,0.0,0.0,0.0,0.0,1002,0,0.0,22.3,22.6,NaN


## 1) Summary stats & missing values

In [4]:
# Summary stats
display(df.describe(include='all'))

# Missing values report
na_counts = df.isna().sum().sort_values(ascending=False)
na_pct = (na_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing": na_counts, "pct": na_pct})
display(missing_report)

,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,WSstdev,WD,WDstdev,BP,Cleaning,Precipitation,TModA,TModB,Comments
count,525600,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,525600.000000,0.0
mean,2022-04-30 12:00:30.000000768,201.957515,116.376337,113.720571,206.643095,198.114691,26.319394,79.448857,1.146113,1.691606,0.363823,133.044668,7.172220,999.876469,0.000967,0.004806,32.504263,32.593091,NaN
min,2021-10-30 00:01:00,-19.500000,-7.800000,-17.900000,0.000000,0.000000,12.300000,9.900000,0.000000,0.000000,0.000000,0.000000,0.000000,993.000000,0.000000,0.000000,10.700000,11.100000,NaN
25%,2022-01-29 06:00:45,-2.800000,-0.300000,-3.800000,0.000000,0.000000,23.100000,68.700000,0.000000,0.000000,0.000000,0.000000,0.000000,999.000000,0.000000,0.000000,23.500000,23.800000,NaN
50%,2022-04-30 12:00:30,0.300000,-0.100000,-0.100000,3.600000,3.400000,25.300000,85.400000,0.800000,1.600000,0.400000,161.500000,6.200000,1000.000000,0.000000,0.000000,26.600000,26.900000,NaN
75%,2022-07-30 18:00:15,362.400000,107.000000,224.700000,359.500000,345.400000,29.400000,96.700000,2.000000,2.600000,0.600000,234.100000,12.000000,1001.000000,0.000000,0.000000,40.900000,41.300000,NaN
max,2022-10-30 00:00:00,1499.000000,946.000000,892.000000,1507.000000,1473.000000,39.900000,100.000000,19.200000,23.900000,4.100000,360.000000,98.400000,1006.000000,1.000000,2.400000,72.800000,70.400000,NaN
std,NaN,298.495150,218.652659,158.946032,300.896893,288.889073,4.398605,20.520775,1.239248,1.617053,0.295000,114.284792,7.535093,2.104419,0.031074,0.047556,12.434899,12.009161,NaN


,missing,pct
Comments,525600,100.0
GHI,0,0.0
Timestamp,0,0.0
DNI,0,0.0
DHI,0,0.0
ModB,0,0.0
ModA,0,0.0
RH,0,0.0
WS,0,0.0
WSgust,0,0.0


## 2) Outlier detection & basic cleaning

In [5]:
numeric_cols = ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"]
df_z = zscore_flags(df, numeric_cols, thresh=3.0)
# Flag counts
flags = {c: int(df_z[f"{c}_outlier"].sum()) for c in numeric_cols if f"{c}_outlier" in df_z.columns}
flags

{'GHI': 2477,
 'DNI': 7586,
 'DHI': 2986,
 'ModA': 1604,
 'ModB': 2041,
 'WS': 3967,
 'WSgust': 3665}

In [ ]:
# Impute median for key columns
key_cols = ["GHI","DNI","DHI","ModA","ModB","Tamb","RH","WS","WSgust","BP","TModA","TModB"]
df_clean = impute_median(df, [c for c in key_cols if c in df.columns])
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH}")

## 3) Time series analysis

In [ ]:
for col in ["GHI","DNI","DHI","Tamb"]:
    if col in df.columns:
        plt.figure()
        plt.plot(df["Timestamp"], df[col])
        plt.title(col)
        plt.xlabel("Timestamp")
        plt.ylabel(col)
        plt.show()

# Monthly patterns
df["month"] = df["Timestamp"].dt.to_period("M")
monthly = df.groupby("month")[["GHI","DNI","DHI","Tamb"]].mean(numeric_only=True)
display(monthly.tail())

## 4) Cleaning impact (ModA/ModB)

In [ ]:
if "Cleaning" in df.columns:
    grp = df.groupby("Cleaning")[["ModA","ModB"]].mean(numeric_only=True)
    display(grp)
    for c in ["ModA","ModB"]:
        if c in df.columns:
            plt.figure()
            grp_plot = grp[c]
            grp_plot.plot(kind="bar")
            plt.title(f"Average {c} by Cleaning flag")
            plt.xlabel("Cleaning flag")
            plt.ylabel(c)
            plt.show()

## 5) Correlation & relationships

In [ ]:
corr_cols = [c for c in ["GHI","DNI","DHI","TModA","TModB","Tamb","RH","WS","WSgust"] if c in df.columns]
if corr_cols:
    corr_mat = df[corr_cols].corr(numeric_only=True)
    display(corr_mat)

    # Simple heatmap with matplotlib
    plt.figure()
    plt.imshow(corr_mat, interpolation='nearest')
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.colorbar()
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# Scatter plots
pairs = [("WS","GHI"),("WSgust","GHI"),("WD","GHI"),("RH","Tamb"),("RH","GHI")]
for x, y in pairs:
    if x in df.columns and y in df.columns:
        plt.figure()
        plt.scatter(df[x], df[y], alpha=0.3)
        plt.xlabel(x); plt.ylabel(y); plt.title(f"{x} vs {y}")
        plt.show()

## 6) Wind & distributions

In [ ]:
# Histograms
for col in ["GHI","WS"]:
    if col in df.columns:
        plt.figure()
        plt.hist(df[col].dropna(), bins=30)
        plt.title(f"Histogram of {col}")
        plt.xlabel(col); plt.ylabel("Count")
        plt.show()

# Wind rose (requires 'windrose' package)
try:
    from windrose import WindroseAxes
    if "WS" in df.columns and "WD" in df.columns:
        ax = WindroseAxes.from_ax()
        ax.bar(df["WD"].values, df["WS"].values, normed=True, opening=0.8, edgecolor='white')
        ax.set_legend()
        plt.show()
except Exception as e:
    print("Windrose not available or failed:", e)

## 7) Bubble chart

In [ ]:
x, y, size = "GHI", "Tamb", "RH" if "RH" in df.columns else ("GHI","Tamb","BP")
if all(c in df.columns for c in [x, y, size]):
    plt.figure()
    plt.scatter(df[x], df[y], s=(df[size].fillna(0) + 1), alpha=0.3)
    plt.xlabel(x); plt.ylabel(y); plt.title(f"{x} vs {y} (size={size})")
    plt.show()